# 铁冠诸侯 · AI 执政官（zkIII-DanielChang-RegentAgent）

> 基于 HelloAgents 框架的智能体，通过浏览器桥接层操控网页游戏《铁冠诸侯》(Lords of Iron Crowns)，在回合制大战略世界中自主决策：发展经济、征兵作战、吞并邻国。

- 游戏：`../index.html`（单文件 JS 大战略游戏，内置 `window.GameAgent` 桥接 API）
- 智能体：`SimpleAgent`（HelloAgents 1.0.0）+ 12 个强类型工具（game_state / end_turn / raise_army / move_army / build / declare_war / …）
- 桥接：Playwright 无头 Edge（工作线程内运行，Jupyter/脚本通用，无需下载浏览器）
- 兜底：LLM 不可用时自动切换零 API 的策略控制器演示，全程不中断

**运行方式**：先 `pip install -r requirements.txt` 并在 `.env` 中配置 LLM 密钥（可选），然后从第 1 个代码单元顺序执行到底。

In [1]:
# ========================================
# 第 1 步：环境准备（路径 / 密钥 / LLM 预检）
# ========================================
import sys
from pathlib import Path

# Jupyter 默认工作目录 = 本 notebook 所在目录，确保 src 包可导入
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from dotenv import load_dotenv
load_dotenv(NOTEBOOK_DIR / ".env", override=True)

# LLM 预检：ping 不通或返回空内容时降级为策略控制器（零 API 成本，演示照常）
llm = None
try:
    from hello_agents import HelloAgentsLLM
    _candidate = HelloAgentsLLM()
    _pong = "".join(_candidate.think([{"role": "user", "content": "只回复两个字：在线"}]))
    if _pong.strip():
        llm = _candidate
        print("LLM 就绪:", _pong.strip()[:20])
    else:
        print("LLM 返回空内容，将使用策略控制器演示")
except Exception as e:
    print("LLM 不可用，将使用零 API 策略控制器演示:", type(e).__name__)

from src.game_bridge import GameBridge
from src.game_agent import CampaignController

🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
在线
LLM 就绪: 在线


In [2]:
# ========================================
# 第 2 步：启动游戏（工作线程内的无头 Edge）
# ========================================
# seed 固定 → 整局可复现；player_id 指定玩家所控国家
bridge = GameBridge().start(seed=20260901, player_id=0)
snap = bridge.get_state()
print("游戏已启动:", bridge.url)
print("开局国家:", snap["player"]["name"], "| 领地", snap["player"]["provs"], "省 | 国库", snap["player"]["treasury"], "金")

游戏已启动: http://127.0.0.1:53032/index.html?agent=1
开局国家: 艾尔多利亚 | 领地 10 省 | 国库 3560 金


In [3]:
# ========================================
# 第 3 步：装配控制器（LLM 智能体 或 策略基线）
# ========================================
controller = CampaignController(bridge, llm, max_tool_iterations=12)
print("控制器就绪，模式:", "LLM 智能体" if llm else "策略基线（零 API）")

✅ 工具 'game_state' 已注册。
✅ 工具 'end_turn' 已注册。
✅ 工具 'raise_army' 已注册。
✅ 工具 'rally_all' 已注册。
✅ 工具 'move_army' 已注册。
✅ 工具 'disband_army' 已注册。
✅ 工具 'build' 已注册。
✅ 工具 'declare_war' 已注册。
✅ 工具 'white_peace' 已注册。
✅ 工具 'enforce_demands' 已注册。
✅ 工具 'resolve_event' 已注册。
✅ 工具 'note_strategy' 已注册。
控制器就绪，模式: LLM 智能体


In [4]:
# ========================================
# 第 4 步：运行 12 回合战役
# ========================================
# 每回合：投喂局势观测 → 行动工具调用 → 结束回合（多重兜底，永不卡死）
# 每 6 回合自动截图一张，完整轨迹写入 outputs/campaign.jsonl
rows = controller.run_campaign(max_turns=12, screenshot_every=6)
print("战役结束")

[回合 1/12] 1066年 2月 领地10省 国库 3275 人力   18 战争0场 动作6次
[回合 2/12] 1066年 3月 领地10省 国库 3191 人力   97 战争0场 动作4次
[回合 3/12] 1066年 4月 领地10省 国库 2936 人力   21 战争0场 动作5次
[回合 4/12] 1066年 5月 领地10省 国库 3281 人力   20 战争0场 动作5次
[回合 5/12] 1066年 6月 领地10省 国库 2842 人力   26 战争0场 动作5次
[回合 6/12] 1066年 7月 领地10省 国库 3203 人力   21 战争0场 动作5次
[回合 7/12] 1066年 8月 领地10省 国库 2981 人力   20 战争0场 动作5次
[回合 8/12] 1066年 9月 领地10省 国库 2867 人力   29 战争0场 动作5次
[回合 9/12] 1066年10月 领地10省 国库 3253 人力   20 战争0场 动作5次
[回合 10/12] 1066年11月 领地10省 国库 3640 人力   21 战争0场 动作3次
[回合 11/12] 1066年12月 领地10省 国库 4027 人力   21 战争0场 动作3次
[回合 12/12] 1067年 1月 领地10省 国库 4415 人力   20 战争0场 动作4次
战役结束


In [5]:
# ========================================
# 第 5 步：战果摘要
# ========================================
for r in rows:
    p = r["player"]
    acts = ", ".join((a["action"] + ("" if a["ok"] else "✗")) for a in r["actions"]) or "无"
    flags = ("[兜底]" if (r.get("forced_end") or r.get("forced_event_choice")) else "")
    print(f"{r['year']}年{r['month']:>2}月 | 领地{p['provinces']:>2}省 | 国库{p['treasury']:>5} | 人力{p['manpower']:>5} | 战争{len(r['wars'])}场 | 动作: {acts} {flags}")

print("\n完整轨迹（含 AI 每回合决策原文）: outputs/campaign.jsonl")
print("截图: outputs/turn_*.png")

1066年 2月 | 领地10省 | 国库 3275 | 人力   18 | 战争0场 | 动作: build, build, raiseArmy, rallyAll, moveArmy, endTurn 
1066年 3月 | 领地10省 | 国库 3191 | 人力   97 | 战争0场 | 动作: build✗, build, moveArmy, endTurn 
1066年 4月 | 领地10省 | 国库 2936 | 人力   21 | 战争0场 | 动作: build, build, raiseArmy, moveArmy, endTurn 
1066年 5月 | 领地10省 | 国库 3281 | 人力   20 | 战争0场 | 动作: build✗, build✗, raiseArmy, moveArmy, endTurn 
1066年 6月 | 领地10省 | 国库 2842 | 人力   26 | 战争0场 | 动作: build, build, raiseArmy, moveArmy, endTurn 
1066年 7月 | 领地10省 | 国库 3203 | 人力   21 | 战争0场 | 动作: build✗, build✗, raiseArmy, moveArmy, endTurn 
1066年 8月 | 领地10省 | 国库 2981 | 人力   20 | 战争0场 | 动作: build, build, raiseArmy, moveArmy, endTurn 
1066年 9月 | 领地10省 | 国库 2867 | 人力   29 | 战争0场 | 动作: build, raiseArmy, moveArmy, build✗, endTurn 
1066年10月 | 领地10省 | 国库 3253 | 人力   20 | 战争0场 | 动作: raiseArmy, moveArmy, build✗, build✗, endTurn 
1066年11月 | 领地10省 | 国库 3640 | 人力   21 | 战争0场 | 动作: raiseArmy, moveArmy, endTurn 
1066年12月 | 领地10省 | 国库 4027 | 人力   21 | 战争0场 | 动作: raiseArmy, moveAr

In [ ]:
# ========================================
# 第 6 步：关闭游戏桥（释放浏览器与端口）
# ========================================
bridge.close()
print("游戏桥已关闭")

## 总结与扩展

- **实现的功能**：智能体通过 12 个强类型工具完整接管游戏——每回合观察局势（含合法移动列表与战斗胜率预测）、执行征召/集结/移动/宣战/建造/议和等全部玩家动作、处理随机事件弹窗，并以 JSONL 轨迹 + 截图的形式留档。
- **遇到的挑战**：事件弹窗的内联闭包无法从外部触发（为 Modal 组件增加 choose/pending 接口解决）；游戏动作全部以对象引用为参数（桥接层统一做 id→对象 解析）；Jupyter 内核的运行中事件循环会让 Playwright 同步 API 拒绝启动（桥接层整体移入工作线程解决）；`Math.random` 导致的撤退不确定性（改为确定性 rng，保证回放一致）。
- **未来改进**：接入视觉输入（截图给多模态模型）；战后复盘写入笔记形成跨战役学习；探索 ReAct/Reflection 范式对比；支持玩家中途接管（headless=False 实景观看）。